# DEPENDENCY SETUP AND MANAGEMENT

In [1]:
import os
import sys
import csv
import re

IN_COLAB = 'google.colab' in str(get_ipython())
if IN_COLAB:
  from google.colab import files

try:
  import pandas as pd
except ImportError as e:
  !pip install pandas
  import pandas as pd

try:
  from tqdm import tqdm
except ImportError as e:
  !pip install tqdm
  from tqdm import tqdm

try:
  from Bio import SeqIO
  from Bio.SeqRecord import SeqRecord
  from Bio.Seq import Seq
except ImportError as e:
  !pip install biopython
  from Bio import SeqIO
  from Bio.SeqRecord import SeqRecord
  from Bio.Seq import Seq

try:
  from rapidfuzz import fuzz, process
except ImportError as e:
  !pip install rapidfuzz
  from rapidfuzz import fuzz, process

# DATA LOADING: METADATA AND FASTA FILES

In [2]:
# CHIKV

# Minimum sequence length threshold
MIN_SEQUENCE_LENGTH = 7000

# GenBank FASTA and METADATA
genbank_fasta = 'sequences.fasta'
genbank_metadata = 'sequences.csv'

# GISAID FASTA and METADATA
gisaid_fasta = 'gisaid_arbo_2026_03_29_06.fasta'
gisaid_metadata = 'gisaid_arbo_2026_03_29_06_Patient_status_metadata.tsv'

# GENBANK METADATA PROCESSING

In [3]:
genbank_df = pd.read_csv(genbank_metadata, sep=',')

# Filter sequences based on selected size
genbank_df = genbank_df[genbank_df['Length'] >= MIN_SEQUENCE_LENGTH].copy()

def parse_isolate_identifier(gb_title):
  """Extract isolate identifier from GenBank title."""
  if pd.isna(gb_title):
    return None

  identifier_patterns = [
      r'\b([A-Za-z]{2,10}(?:\s+virus)?/[\w\s\.\-]+/[\w]{2,10}/[^/\s,;\)]+(?:/\d{4})?)',
      r'\b([A-Za-z]{2,10}(?:\s+virus)?/[\w\s\.\-]+/[\w]{2,10}/[\d]{4}/[^,\s\)]+)',
      r'\b((?:[A-Za-z][A-Za-z0-9_\-\.\s]+/){2,}[A-Za-z0-9_\-\.]+)',
      r'/([A-Z]{2,3})/\d{4}/([^/,\s;\)]+)',
      r'\b([A-Z]{2,5}[-_]\d{3,5}(?:[-_][A-Z]{2,4})?)\b',
      r'\b([A-Z]{2,5}\d{2,5}[\w_\-]*)\b',
      r'\b([A-Z]{2,4}\s+\d+[\w\-]+)\b',
      r'\b([A-Z]{2,5}[\d\w_\-]*_\d{4}-\d{2}-\d{2})\b',
      r'(?:isolate)[:\s]+([A-Z]{2,4}\s+[\w\-]+\d+[\w\-]*)',
      r'(?:isolate)[:\s]+([^\s,;\(\)]+)',
      r'/\b([A-Za-z]{2,}[\dA-Za-z_\-\.]+)\b',
      r'\b([A-Z]{2}[\w_\-]{3,20})\b']

  for pattern in identifier_patterns:
    match = re.search(pattern, gb_title, re.I)
    if match:
      result = match.group(1).strip()
      if len(result) >= 2:
        return result

  return None

genbank_df['Isolate_Title'] = genbank_df['GenBank_Title'].apply(parse_isolate_identifier)

# Handle missing values
genbank_df['Isolate_Title'] = genbank_df['Isolate_Title'].fillna('')

# Process geographic location
genbank_df['Location'] = genbank_df['Geo_Location'].apply(
    lambda x: x.split(':', 1)[1].strip()
    if pd.notna(x) and ':' in str(x) else "").apply(
        lambda x: str(x).
        replace('/', ', ').
        replace('(', ', ').
        replace(')', '').
        replace('-', ', ').strip()
    if pd.notna(x) else "").apply(
        lambda x: ', '.join([part.strip()
        for part in str(x).split(',') if part.strip()])
    if pd.notna(x) and str(x).strip() else "")
genbank_df['Location'] = genbank_df['Location'].apply(
    lambda x: [part.strip() for part in x.split(',')]
    if pd.notna(x) and str(x).strip() else [])
for i in range(4):
  genbank_df[f'Location_Level{i+1}'] = genbank_df['Location'].apply(
      lambda x, idx=i: x[idx] if isinstance(x, list) and len(x) > idx else "")
genbank_df = genbank_df.drop('Location', axis=1)

# Standardize location names
genbank_df['Country'] = genbank_df['Country'].replace({
    "Cape Verde": "Cabo Verde",
    "Democratic Republic of the Congo": "Republic of the Congo",
    "La Reunion": "Reunion",
    "La Reunion": "La La Reunion",
    "US Virgin Islands": "United States of America",
    "USA": "United States of America",
    "Viet Nam": "Vietnam"})
for col in ['Country', 'Location_Level1', 'Location_Level2', 'Location_Level3', 'Location_Level4']:
  genbank_df[col] = genbank_df[col].apply(
      lambda x: ''.join(word.capitalize() for word in re.split(r'[,\s\-_/\(\)\.;:]+', str(x)) if word)
      if pd.notna(x) and str(x).strip() else "")

# Standardize date collection to YYYY-MM-DD format
genbank_df['Collection_Date'] = genbank_df['Collection_Date'].astype(str)
genbank_df['Collection_Date'] = genbank_df['Collection_Date'].str.replace('.0', '')
genbank_df['Collection_Date'] = genbank_df['Collection_Date'].apply(lambda x: (
    "" if x in ['nan', ''] else
    f"{x}-XX-XX" if len(x) == 4 else
    f"{x}-XX" if len(x) == 7 else x))

# Prefix column names for source identification
genbank_df.columns = ['GenBank_' + col if col not in ['GenBank_Accession',
                                                      'GenBank_RefSeq',
                                                      'GenBank_Title']
                      else col for col in genbank_df.columns]

# genbank_df.to_csv('genbank_df.csv', index=False, quoting=csv.QUOTE_ALL)

# GISAID METADATA PROCESSING

In [4]:
gisaid_df = pd.read_csv(gisaid_metadata, sep='\t')

# Filter sequences based on selected size
gisaid_df = gisaid_df[gisaid_df['Sequence Length'] >= MIN_SEQUENCE_LENGTH].copy()

gisaid_df.rename(columns={'Accession ID': 'Accession',
                          'Virus name': 'Isolate',
                          'Collection date': 'Collection_Date',
                          'Additional location information': 'Additional_Location_Info',
                          'Sampling strategy': 'Sampling_Strategy',
                          'Gender': 'Host_Gender',
                          'Patient age': 'Host_Age',
                          'Patient status': 'Host_Status',
                          'Last vaccinated': 'Host_Last_Vaccinated',
                          'Additional host information': 'Additional_Host_Info',
                          'AA Substitutions': 'AA_Substitutions',
                          'Sequence Length': 'Length'}, inplace=True)

def parse_isolate_components(isolate_name):
  """Parse and extract isolate identifier from GISAID Isolate."""
  if pd.notna(isolate_name):
    components = isolate_name.split('/')
    if len(components) >= 2:
      return '/'.join(components[-2:])
  return None

gisaid_df['Isolate'] = gisaid_df['Isolate'].apply(parse_isolate_components)

# Handle missing values
gisaid_df['Isolate'] = gisaid_df['Isolate'].fillna('')

# Standardize host taxonomy
gisaid_df['Host'] = gisaid_df['Host'].str.replace('Other: ', '', case=False)
gisaid_df['Host'] = gisaid_df['Host'].replace({"Aedes sp": "Aedes sp.",
                                               "Culex sp": "Culex sp.",
                                               "Human": "Homo sapiens",
                                               "unknown": ""})

# Process geographic location
gisaid_df['Geo_Location'] = gisaid_df['Location']
gisaid_df['Location'] = gisaid_df['Location'].apply(
    lambda x: [part.strip() for part in str(x).split('/')]
    if pd.notna(x) else [])
gisaid_df['Country'] = gisaid_df['Location'].apply(
    lambda x: x[1] if isinstance(x, list) and len(x) > 1 else "")
for i in range(1, 5):
  gisaid_df[f'Location_Level{i}'] = gisaid_df['Location'].apply(
      lambda x, idx=i+1: x[idx] if isinstance(x, list) and len(x) > idx else "")
gisaid_df = gisaid_df.drop('Location', axis=1)

# Standardize location names
gisaid_df['Country'] = gisaid_df['Country'].replace({
    "Cape Verde": "Cabo Verde",
    "Democratic Republic of the Congo": "Republic of the Congo",
    "La Reunion": "Reunion",
    "La Reunion": "La La Reunion",
    "US Virgin Islands": "United States of America",
    "USA": "United States of America",
    "Viet Nam": "Vietnam"})
for col in ['Country', 'Location_Level1', 'Location_Level2', 'Location_Level3', 'Location_Level4']:
  gisaid_df[col] = gisaid_df[col].apply(
      lambda x: ''.join(word.capitalize() for word in re.split(r'[,\s\-_/\(\)\.;:]+', str(x)) if word)
      if pd.notna(x) and str(x).strip() else "")

# Standardize date collection to YYYY-MM-DD format
gisaid_df['Collection_Date'] = gisaid_df['Collection_Date'].replace('unknown', '')
gisaid_df['Collection_Date'] = gisaid_df['Collection_Date'].astype(str)
gisaid_df['Collection_Date'] = gisaid_df['Collection_Date'].str.replace('.0', '')
gisaid_df['Collection_Date'] = gisaid_df['Collection_Date'].apply(lambda x: (
    "" if x in ['nan', ''] else
    f"{x}-XX-XX" if len(x) == 4 else
    f"{x}-XX" if len(x) == 7 else x))

# Prefix column names for source identification
gisaid_df.columns = ['GISAID_' + col if col != 'GISAID_Accession'
else col for col in gisaid_df.columns]

# gisaid_df.to_csv('gisaid_df.csv', index=False, quoting=csv.QUOTE_ALL)

# FASTA-BASED MATCHING

In [5]:
genbank_fasta_filt = []
for record in SeqIO.parse(genbank_fasta, "fasta"):
  if len(record.seq) >= MIN_SEQUENCE_LENGTH:
    genbank_fasta_filt.append(record)

gisaid_fasta_filt = []
for record in SeqIO.parse(gisaid_fasta, "fasta"):
  if len(record.seq) >= MIN_SEQUENCE_LENGTH:
    gisaid_fasta_filt.append(record)

SeqIO.write(genbank_fasta_filt, "genbank_fasta_filt.fasta", "fasta")
SeqIO.write(gisaid_fasta_filt, "gisaid_fasta_filt.fasta", "fasta")

def read_fasta_to_dict(file_path, extract_epi_isl=False):
  """Parses FASTA file and return dictionary mapping nucleotide sequences to their accession identifiers."""
  sequences_registry = {}
  for record in SeqIO.parse(file_path, "fasta"):
    sequence = str(record.seq).replace(" ", "").replace("\n", "").upper()
    if extract_epi_isl and '|' in record.id:
      sequence_id = record.id.split('|')[1]
    else:
      sequence_id = record.id
    sequences_registry.setdefault(sequence, []).append(sequence_id)
  return sequences_registry

genbank_sequences = read_fasta_to_dict("genbank_fasta_filt.fasta")
gisaid_sequences = read_fasta_to_dict("gisaid_fasta_filt.fasta", extract_epi_isl=True)

identical_sequences = {}
for seq in genbank_sequences:
  if seq in gisaid_sequences:
    identical_sequences[seq] = {'GenBank_Accession': genbank_sequences[seq],
                                'GISAID_Accession': gisaid_sequences[seq]}

data = []
for seq, ids in identical_sequences.items():
  for genbank_id in ids['GenBank_Accession']:
    for gisaid_id in ids['GISAID_Accession']:
      data.append({'GenBank_Accession': genbank_id,
                   'GISAID_Accession': gisaid_id,
                   'Sequence': seq, })
matches_fasta_df = pd.DataFrame(data)

matches_fasta_df['GISAID_Accession'] = matches_fasta_df['GISAID_Accession'].astype(str)

# ISOLATE FIELD-BASED MATCHING (FUZZY MATCHING)

In [6]:
genbank_isolate_df = genbank_df[['GenBank_Accession', 'GenBank_Isolate', 'GenBank_Isolate_Title']].copy()
gisaid_isolate_df = gisaid_df[['GISAID_Accession', 'GISAID_Isolate']].copy()

def fuzzy_identifier_matching(source_df, target_df, source_col, target_col, similarity_threshold, result_limit=5):
  """Perform fuzzy string matching between isolate identifier fields."""
  source_df[source_col] = source_df[source_col].fillna('').astype(str).str.lower()
  target_df[target_col] = target_df[target_col].fillna('').astype(str).str.lower()

  target_choices = target_df[target_col].tolist()
  target_dict = {i: row for i, row in enumerate(target_df.to_dict('records'))}

  best_matches = {}

  for idx, row in tqdm(source_df.iterrows(), total=len(source_df), desc="Fuzzy matching"):
    identifier = row[source_col]
    if not identifier or len(identifier) < 3:
      continue

    potential_matches = process.extract(identifier, target_choices,
                                        scorer=fuzz.partial_ratio,
                                        limit=result_limit,
                                        score_cutoff=similarity_threshold)

    for match_identifier, similarity_score, match_idx in potential_matches:
      if similarity_score >= similarity_threshold:
        matched_record = target_dict[match_idx]
        match_data = {
            'GenBank_Accession': row['GenBank_Accession'],
            'GISAID_Accession': matched_record['GISAID_Accession'],
            'GenBank_Isolate': row['GenBank_Isolate'],
            'GenBank_Isolate_Title': row['GenBank_Isolate_Title'],
            'GISAID_Isolate': matched_record['GISAID_Isolate'],
            'Similarity_Score': similarity_score}

        genbank_acc = row['GenBank_Accession']
        if genbank_acc not in best_matches:
          best_matches[genbank_acc] = match_data
        elif similarity_score > best_matches[genbank_acc]['Similarity_Score']:
          best_matches[genbank_acc] = match_data

  match_results = list(best_matches.values())

  return match_results

matches_isolate_df = pd.DataFrame(fuzzy_identifier_matching(genbank_isolate_df, gisaid_isolate_df, 'GenBank_Isolate', 'GISAID_Isolate', 80))
matches_isolate_ttl_df = pd.DataFrame(fuzzy_identifier_matching(genbank_isolate_df, gisaid_isolate_df, 'GenBank_Isolate_Title', 'GISAID_Isolate', 80))

Fuzzy matching: 100%|██████████████████████| 6432/6432 [00:52<00:00, 123.46it/s]


# METADATA-BASED MATCHING (LOCATION, COLLECTION DATE, AND SEQUENCE LENGTH)

In [7]:
genbank_lcdl_df = genbank_df[['GenBank_Accession',
                              'GenBank_Country',
                              'GenBank_Location_Level1',
                              'GenBank_Location_Level2',
                              'GenBank_Collection_Date',
                              'GenBank_Length']].copy()

gisaid_lcdl_df = gisaid_df[['GISAID_Accession',
                            'GISAID_Country',
                            'GISAID_Location_Level1',
                            'GISAID_Location_Level2',
                            'GISAID_Collection_Date',
                            'GISAID_Length']].copy()

# Handle missing values
for col in ['GenBank_Country',
            'GenBank_Location_Level1',
            'GenBank_Location_Level2',
            'GenBank_Collection_Date',
            'GenBank_Length']:
  genbank_lcdl_df[col] = genbank_lcdl_df[col].fillna('').astype(str).str.lower()
for col in ['GISAID_Country',
            'GISAID_Location_Level1',
            'GISAID_Location_Level2',
            'GISAID_Collection_Date',
            'GISAID_Length']:
  gisaid_lcdl_df[col] = gisaid_lcdl_df[col].fillna('').astype(str).str.lower()

# Perform exact metadata matching
matches_lcdl_df = pd.merge(genbank_lcdl_df, gisaid_lcdl_df,
                           how='inner',
                           left_on=['GenBank_Country',
                                    'GenBank_Location_Level1',
                                    'GenBank_Location_Level2',
                                    'GenBank_Collection_Date',
                                    'GenBank_Length'],
                           right_on=['GISAID_Country',
                                     'GISAID_Location_Level1',
                                     'GISAID_Location_Level2',
                                     'GISAID_Collection_Date',
                                     'GISAID_Length'])

# DATA INTEGRATION MERGING

In [8]:
integration_strategies = [
    (matches_fasta_df[['GenBank_Accession', 'GISAID_Accession']], 'FASTA'),
    (matches_isolate_df[['GenBank_Accession', 'GISAID_Accession']], 'ISOLATE'),
    (matches_isolate_ttl_df[['GenBank_Accession', 'GISAID_Accession']], 'ISOLATE_TITLE'),
    (matches_lcdl_df[['GenBank_Accession', 'GISAID_Accession']], 'LCDL')]

integration_results = []
for match_df, strategy_name in integration_strategies:
  df_copy = match_df.copy()
  df_copy['Integration_Method'] = strategy_name
  integration_results.append(df_copy)
all_integration_records = pd.concat(integration_results, ignore_index=True)

# Aggregate by accession pair
consolidated_matches = all_integration_records.groupby(['GenBank_Accession', 'GISAID_Accession']).agg({
    'Integration_Method': lambda x: ', '.join(sorted(x))}).reset_index()

consolidated_matches['Confidence_Score'] = consolidated_matches['Integration_Method'].apply(lambda x: len(x.split(', ')))

consolidated_matches = consolidated_matches.sort_values(['Confidence_Score', 'Integration_Method'],
                                                        ascending=[False, True])

# Create accession-to-sequence mappings
genbank_accession_to_sequence = {}
for seq, accessions in genbank_sequences.items():
  for acc in accessions:
    genbank_accession_to_sequence[acc] = seq
gisaid_accession_to_sequence = {}
for seq, accessions in gisaid_sequences.items():
  for acc in accessions:
    gisaid_accession_to_sequence[acc] = seq

# Get all accessions from consolidated matches
genbank_matched_accs = set(consolidated_matches['GenBank_Accession'])
gisaid_matched_accs = set(consolidated_matches['GISAID_Accession'])

# Get all accessions from source datasets
genbank_all_accs = set(genbank_df['GenBank_Accession'])
gisaid_all_accs = set(gisaid_df['GISAID_Accession'])

# Identify unmatched accessions
genbank_unmatched_accs = genbank_all_accs - genbank_matched_accs
gisaid_unmatched_accs = gisaid_all_accs - gisaid_matched_accs

# Create DataFrame and add metadata for POTENTIAL_MATCH
potential_match_df = consolidated_matches.copy()
potential_match_df['Sequence'] = potential_match_df['GenBank_Accession'].map(genbank_accession_to_sequence)
potential_match_df['Integration_Status'] = 'POTENTIAL_MATCH'
potential_match_df = pd.merge(potential_match_df, genbank_df,
                              on='GenBank_Accession', how='left')
potential_match_df = pd.merge(potential_match_df, gisaid_df,
                              on='GISAID_Accession', how='left')

# Create DataFrame for GENBANK_ONLY records
genbank_only_df = genbank_df[genbank_df['GenBank_Accession'].isin(genbank_unmatched_accs)].copy()
genbank_only_df['GISAID_Accession'] = ''
genbank_only_df['Integration_Method'] = 'NO_INTEGRATION'
genbank_only_df['Confidence_Score'] = 0
genbank_only_df['Sequence'] = genbank_only_df['GenBank_Accession'].map(genbank_accession_to_sequence)
genbank_only_df['Integration_Status'] = 'GENBANK_ONLY'

# Create DataFrame for GISAID_ONLY records
gisaid_only_df = gisaid_df[gisaid_df['GISAID_Accession'].isin(gisaid_unmatched_accs)].copy()
gisaid_only_df['GenBank_Accession'] = ''
gisaid_only_df['Integration_Method'] = 'NO_INTEGRATION'
gisaid_only_df['Confidence_Score'] = 0
gisaid_only_df['Sequence'] = gisaid_only_df['GISAID_Accession'].map(gisaid_accession_to_sequence)
gisaid_only_df['Integration_Status'] = 'GISAID_ONLY'

# Combine POTENTIAl_MATCH, GENBANK_ONLY and GISAID_ONLY DataFrames
merged_df = pd.concat([potential_match_df, genbank_only_df, gisaid_only_df], ignore_index=True)

match_records = merged_df[~merged_df['Integration_Status'].isin(['GENBANK_ONLY', 'GISAID_ONLY'])].copy()
match_records['wFASTA'] = match_records['Integration_Method'].str.contains('FASTA', na=False)

match_records['Integration_Status'] = match_records.apply(
    lambda row: 'HIGH_POTENTIAL_MATCH' if row['wFASTA'] else row['Integration_Status'], axis=1)

# Selects optimal GISAID match for each GenBank record based on confidence hierarchy
genbank_matches = []
for genbank_id, group in match_records.groupby('GenBank_Accession'):
  total_matches = len(group)
  if total_matches == 1:
    single_match = group.iloc[0].copy()
    if single_match['Integration_Status'] == 'HIGH_POTENTIAL_MATCH':
      single_match['Integration_Status'] = 'MATCH'
    genbank_matches.append(pd.DataFrame([single_match]))
  else:
    # Separate matches by confidence level
    high_potential = group[group['Integration_Status'] == 'HIGH_POTENTIAL_MATCH']
    potential = group[group['Integration_Status'] == 'POTENTIAL_MATCH']
    if len(high_potential) > 0:
      # Prioritize matches with sequence identity evidence
      high_potential = high_potential.sort_values('Confidence_Score', ascending=False)
      best_score = high_potential.iloc[0]['Confidence_Score']
      best_matches = high_potential[high_potential['Confidence_Score'] == best_score].copy()
      if len(best_matches) == 1:
        best_matches.iloc[0, best_matches.columns.get_loc('Integration_Status')] = 'MATCH'
        genbank_matches.append(best_matches)
      else:
        genbank_matches.append(best_matches)
    elif len(potential) > 0:
      # Fallback to regular potential matches
      potential = potential.sort_values(['Confidence_Score', 'wFASTA'], ascending=[False, False])
      best_score = potential.iloc[0]['Confidence_Score']
      best_matches = potential[potential['Confidence_Score'] == best_score].copy()
      if len(best_matches) == 1:
        best_matches.iloc[0, best_matches.columns.get_loc('Integration_Status')] = 'HIGH_POTENTIAL_MATCH'
        genbank_matches.append(best_matches)
      else:
        genbank_matches.append(best_matches)
    else:
      genbank_matches.append(group.head(1).copy())
genbank_matches_df = pd.concat(genbank_matches, ignore_index=True)

# Selects optimal GenBank match for each GISAID record based on confidence hierarchy
all_gisaid_duplicates = genbank_matches_df[genbank_matches_df.duplicated('GISAID_Accession', keep=False)].copy()
if len(all_gisaid_duplicates) > 0:
  # Define priority hierarchy for match resolution
  status_order = {'MATCH': 1, 'HIGH_POTENTIAL_MATCH': 2, 'POTENTIAL_MATCH': 3}
  all_gisaid_duplicates['Status_Order'] = all_gisaid_duplicates['Integration_Status'].map(status_order)
  all_gisaid_duplicates = all_gisaid_duplicates.sort_values(['Status_Order', 'Confidence_Score', 'wFASTA'], ascending=[True, False, False])
  no_duplicates = genbank_matches_df[~genbank_matches_df['GISAID_Accession'].isin(all_gisaid_duplicates['GISAID_Accession'].unique())]
  best_matches_list = []
  excluded_matches_list = []
  for gisaid_acc, group in all_gisaid_duplicates.groupby('GISAID_Accession'):
    if len(group) == 1:
      best_matches_list.append(group)
    else:
      max_score = group['Confidence_Score'].max()
      max_score_matches = group[group['Confidence_Score'] == max_score]
      if len(max_score_matches) == 1:
        best_matches_list.append(max_score_matches)
        excluded = group[group.index != max_score_matches.index[0]]
      else:
        best_matches_list.append(max_score_matches.head(1))
        excluded = group[group.index != max_score_matches.index[0]]
      # Preserves integration evidence while marking as secondary candidate
      for _, row in excluded.iterrows():
        excluded_row = row.copy()
        excluded_row['GISAID_Accession'] = gisaid_acc
        excluded_row['Integration_Status'] = 'POTENTIAL_MATCH'
        excluded_matches_list.append(pd.DataFrame([excluded_row]))
  best_gisaid_matches = pd.concat(best_matches_list, ignore_index=True) if best_matches_list else pd.DataFrame()
  excluded_gisaid_df = pd.concat(excluded_matches_list, ignore_index=True) if excluded_matches_list else pd.DataFrame()
  gisaid_matches_df = pd.concat([no_duplicates, best_gisaid_matches, excluded_gisaid_df], ignore_index=True)
else:
  gisaid_matches_df = genbank_matches_df.copy()

# Reintegrate GENBANK_ONLY and GISAID_ONLY records excluded from match resolution
best_merged_df = gisaid_matches_df.copy()
if 'wFASTA' in best_merged_df.columns:
  best_merged_df = best_merged_df.drop('wFASTA', axis=1)
only_records = merged_df[merged_df['Integration_Status'].isin(['GENBANK_ONLY', 'GISAID_ONLY'])].copy()
best_merged_df = pd.concat([best_merged_df, only_records], ignore_index=True)

# Filter POTENTIAL_MATCH by sequence length agreement
potential_matches = best_merged_df[best_merged_df['Integration_Status'] == 'POTENTIAL_MATCH'].copy()
if len(potential_matches) > 0:
  potential_matches['GenBank_Length'] = pd.to_numeric(potential_matches['GenBank_Length'], errors='coerce')
  potential_matches['GISAID_Length'] = pd.to_numeric(potential_matches['GISAID_Length'], errors='coerce')
  different_length_mask = (
      (potential_matches['GenBank_Length'] != potential_matches['GISAID_Length']) &
       (~potential_matches['GenBank_Length'].isna()) &
        (~potential_matches['GISAID_Length'].isna()))
  problematic_potential = potential_matches[different_length_mask]
  if len(problematic_potential) > 0:
    new_genbank_only_records = []
    new_gisaid_only_records = []
    for idx, row in problematic_potential.iterrows():
      genbank_acc = row['GenBank_Accession']
      gisaid_acc = row['GISAID_Accession']
      genbank_only_record = row.copy()
      genbank_only_record['GISAID_Accession'] = ''
      genbank_only_record['Integration_Method'] = 'NO_INTEGRATION'
      genbank_only_record['Confidence_Score'] = 0
      genbank_only_record['Integration_Status'] = 'GENBANK_ONLY'
      gisaid_columns = [col for col in genbank_only_record.index if 'GISAID' in col]
      for col in gisaid_columns:
        if col != 'GISAID_Accession':
          genbank_only_record[col] = ''
      new_genbank_only_records.append(genbank_only_record)
      gisaid_only_record = row.copy()
      gisaid_only_record['GenBank_Accession'] = ''
      gisaid_only_record['Integration_Method'] = 'NO_INTEGRATION'
      gisaid_only_record['Confidence_Score'] = 0
      gisaid_only_record['Integration_Status'] = 'GISAID_ONLY'
      genbank_columns = [col for col in gisaid_only_record.index if 'GenBank' in col]
      for col in genbank_columns:
        if col != 'GenBank_Accession':
          gisaid_only_record[col] = ''
      new_gisaid_only_records.append(gisaid_only_record)
    if new_genbank_only_records:
      genbank_problematic_df = pd.DataFrame(new_genbank_only_records)
    if new_gisaid_only_records:
      gisaid_problematic_df = pd.DataFrame(new_gisaid_only_records)
    problematic_indices = problematic_potential.index
    best_merged_df = best_merged_df[~best_merged_df.index.isin(problematic_indices)]
    if new_genbank_only_records and new_gisaid_only_records:
      best_merged_df = pd.concat([best_merged_df, genbank_problematic_df, gisaid_problematic_df], ignore_index=True)

# Detect duplicated GenBank or GISAID accessions requiring data consolidation
indices_to_remove = []
genbank_duplicates = best_merged_df[best_merged_df['GenBank_Accession'].duplicated(keep=False)]['GenBank_Accession'].unique()
for genbank_acc in genbank_duplicates:
  if not genbank_acc:
    continue
  records = best_merged_df[best_merged_df['GenBank_Accession'] == genbank_acc]
  if any(records['Integration_Status'] == 'GENBANK_ONLY'):
    if any(records['Integration_Status'].isin(['MATCH', 'HIGH_POTENTIAL_MATCH', 'POTENTIAL_MATCH'])):
      genbank_only_idx = records[records['Integration_Status'] == 'GENBANK_ONLY'].index
      indices_to_remove.extend(genbank_only_idx.tolist())
gisaid_duplicates = best_merged_df[best_merged_df['GISAID_Accession'].duplicated(keep=False)]['GISAID_Accession'].unique()
for gisaid_acc in gisaid_duplicates:
  if not gisaid_acc:
    continue
  records = best_merged_df[best_merged_df['GISAID_Accession'] == gisaid_acc]
  if any(records['Integration_Status'] == 'GISAID_ONLY'):
    if any(records['Integration_Status'].isin(['MATCH', 'HIGH_POTENTIAL_MATCH', 'POTENTIAL_MATCH'])):
      gisaid_only_idx = records[records['Integration_Status'] == 'GISAID_ONLY'].index
      indices_to_remove.extend(gisaid_only_idx.tolist())
indices_to_remove = list(set(indices_to_remove))
if indices_to_remove:
    before_count = len(best_merged_df)
    best_merged_df = best_merged_df.drop(indices_to_remove)
    after_count = len(best_merged_df)

# Remove sequences derived from experimental methods, diagnostics, and vaccine development
best_merged_df = best_merged_df[~best_merged_df['GenBank_RefSeq'].str.contains('refseq', case=False, na=False)]
best_merged_df = best_merged_df[~best_merged_df['GenBank_Host'].str.contains('macaca|mouse|mus|simiiformes', case=False, na=False)]
best_merged_df = best_merged_df[~best_merged_df['GISAID_Host'].str.contains('macaca|mouse|mus|primates|sentinel', case=False, na=False)]
best_merged_df = best_merged_df[~best_merged_df['GISAID_Passage'].str.contains('A549|AP61|culture|C636|C6/36|mouse|vero', case=False, na=False)]
non_phylogenetics = ['assay', 'artificial', 'attenuated', 'clone',
                     'culture', 'diagnostic', 'fda', 'immunogenic',
                     'immunogenical', 'innoculum', 'method', 'mouse',
                     'mutant', 'paratransgenic', 'patent', 'primer',
                     'prototype', 'purified', 'recombinant', 'rt-pcr',
                     'vaccination', 'vaccine', 'virus like']
for word in non_phylogenetics:
  best_merged_df = best_merged_df[~best_merged_df['GenBank_Title'].astype(str).
                                  str.contains(word, case=False)]

best_merged_df = best_merged_df.drop(columns=['Status_Order'], errors='ignore')

# Define optimal column ordering
preferred_column_order = [
    'GenBank_Accession', 'GISAID_Accession', 'Integration_Status',
    'Integration_Method', 'GenBank_Isolate', 'GenBank_Isolate_Title',
    'GISAID_Isolate', 'GenBank_Country', 'GISAID_Country',
    'GenBank_Collection_Date', 'GISAID_Collection_Date', 'GenBank_Location_Level1',
    'GISAID_Location_Level1', 'GenBank_Location_Level2', 'GISAID_Location_Level2',
    'GenBank_Location_Level3', 'GISAID_Location_Level3', 'GenBank_Location_Level4',
    'GISAID_Location_Level4', 'GenBank_Length', 'GISAID_Length',
    'GenBank_Host', 'GISAID_Host', 'GenBank_Genotype',
    'GISAID_Genotype', 'GISAID_Lineage', 'GenBank_Segment',
    'GenBank_Submitters', 'GenBank_Organization', 'GenBank_Org_location',
    'GenBank_RefSeq', 'GenBank_Assembly', 'GenBank_SRA_Accession',
    'GenBank_BioSample', 'GenBank_BioProject', 'GenBank_Organism_Name',
    'GenBank_Species', 'GenBank_Genus', 'GenBank_Family',
    'GenBank_Title', 'GenBank_Nuc_Completeness', 'GenBank_Geo_Location',
    'GISAID_Geo_Location', 'GenBank_USA', 'GenBank_Tissue_Specimen_Source',
    'GISAID_Specimen', 'GISAID_Passage', 'GenBank_Publications',
    'GenBank_Release_Date', 'GenBank_Molecule_type', 'GISAID_Additional_Location_Info',
    'GISAID_Sampling_Strategy', 'GISAID_Host_Gender', 'GISAID_Host_Age',
    'GISAID_Host_Status', 'GISAID_Host_Last_Vaccinated', 'GISAID_Additional_Host_Info',
    'GISAID_AA_Substitutions', 'Sequence']
new_column_order = []
for col in preferred_column_order:
    if col in best_merged_df.columns:
        new_column_order.append(col)
other_columns = [col for col in best_merged_df.columns if col not in new_column_order]
new_column_order.extend(sorted(other_columns))
best_merged_df = best_merged_df[new_column_order]

best_merged_df = best_merged_df.sort_values(
    by=['Integration_Status', 'Confidence_Score', 'GenBank_Accession', 'GISAID_Accession'],
    ascending=[True, False, True, True])

best_merged_df.to_csv('best_merged_df.csv', index=False, quoting=csv.QUOTE_ALL)

print(f"Sequence records from GenBank: {len(genbank_all_accs)}")
print(f"Sequence records from GISAID: {len(gisaid_all_accs)}")
print(f"\nCount distribution:")
status_counts = best_merged_df['Integration_Status'].value_counts()
for status in ['MATCH', 'HIGH_POTENTIAL_MATCH', 'POTENTIAL_MATCH', 'GENBANK_ONLY', 'GISAID_ONLY']:
  count = status_counts.get(status, 0)
  print(f"- {status}: {count}")
print(f"\nTotal of preprocessing records: {len(best_merged_df)}")

Sequence records from GenBank: 6432
Sequence records from GISAID: 6544

Count distribution:
- MATCH: 2965
- HIGH_POTENTIAL_MATCH: 219
- POTENTIAL_MATCH: 22413
- GENBANK_ONLY: 2211
- GISAID_ONLY: 3117

Total of preprocessing records: 30925


# DOWNLOAD THE PHY VIRAL DATA INTEGRATED DATASET

In [ ]:
if IN_COLAB:
  # files.download('genbank_fasta_filt.fasta')
  # files.download('gisaid_fasta_filt.fasta')
  # files.download('merged_df.csv')
  files.download('best_merged_df.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>